# Customer Segmentation Using RFM Analysis

Analyze customer purchasing behavior using RFM (Recency, Frequency, Monetary) analysis to better understand customer value and segmentation.

**Business Gold**: Identify valuable customers, at-risk customers, and customer purchasing patterns to support targeted marketing and retention strategies.


## 1. Import Libraries

Import libraries for data analysis and visualization.

In [1]:
import pandas as pd

df = pd.read_csv("online_retail.csv")

## 2. Load Dataset

Load customer transaction data and inspect the dataset structure.

In [2]:
# Cleaning
df = df.dropna(subset=["CustomerID", "Description"])
df = df[df["Quantity"] > 0]
df = df[df["UnitPrice"] > 0]

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["Sales"] = df["Quantity"] * df["UnitPrice"]

/var/folders/y0/g_vvng7x6gn1ggql3qv6040m0000gn/T/ipykernel_28557/2897044845.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])


## 3. Data Cleaning

Clean missing values and remove invalid transactions before analysis.

In [3]:
# Snapshot date
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

## 4. Feature Engineering

Create additional variables such as sales-related metrics.

In [4]:
# RFM table
rfm = df.groupby("CustomerID").agg({
    "InvoiceDate": lambda x: (snapshot_date - x.max()).days,
    "InvoiceNo": "nunique",
    "Sales": "sum"
}).reset_index()

rfm.columns = ["CustomerID", "Recency", "Frequency", "Monetary"]

## 5. RFM Calculation

Calculate Recency, Frequency, and Monetary metrics for each customer.

In [5]:
# Score each metric
rfm["R_score"] = pd.qcut(rfm["Recency"], 4, labels=[4, 3, 2, 1])
rfm["F_score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 4, labels=[1, 2, 3, 4])
rfm["M_score"] = pd.qcut(rfm["Monetary"], 4, labels=[1, 2, 3, 4])

rfm["RFM_Score"] = (
    rfm["R_score"].astype(str) +
    rfm["F_score"].astype(str) +
    rfm["M_score"].astype(str)
)

## 6. RFM Scoring

Convert RFM metrics into customer scores for comparison.

In [6]:
# Simple segment rule
def segment_customer(row):
    if row["RFM_Score"] in ["444", "443", "434", "344"]:
        return "VIP"
    elif row["R_score"] in [1, 2]:
        return "At Risk"
    else:
        return "Regular"

rfm["Segment"] = rfm.apply(segment_customer, axis=1)

## 7. Customer Segmentation

Segment customers based on purchasing behavior.

In [7]:
print(rfm.head())

   CustomerID  Recency  Frequency  Monetary R_score F_score M_score RFM_Score  \
0     12346.0      326          1  77183.60       1       1       4       114   
1     12347.0        2          7   4310.00       4       4       4       444   
2     12348.0       75          4   1797.24       2       3       4       234   
3     12349.0       19          1   1757.55       3       1       4       314   
4     12350.0      310          1    334.40       1       1       2       112   

   Segment  
0  At Risk  
1      VIP  
2  At Risk  
3  Regular  
4  At Risk  


## 8. Segment Analysis & Insights

Analyze customer groups and summarize key business insights.

In [8]:
print(rfm["Segment"].value_counts())

Segment
At Risk    2150
Regular    1311
VIP         877
Name: count, dtype: int64


## 9. Business Recommendations

Provide actionable recommendations based on customer behavior.

## 10. Export CSV for Tableau Dashboard

Export segmentation results to CSV files for Tableau visualization.

In [9]:
import os
os.makedirs("../outputs", exist_ok=True)

# Full RFM table with segment labels
rfm.to_csv("../outputs/rfm_segments.csv", index=False)

# Segment-level summary (count, avg RFM, total revenue)
segment_summary = rfm.groupby("Segment").agg(
    customer_count=("CustomerID", "count"),
    avg_recency=("Recency", "mean"),
    avg_frequency=("Frequency", "mean"),
    avg_monetary=("Monetary", "mean"),
    total_revenue=("Monetary", "sum")
).reset_index().round(2)
segment_summary.to_csv("../outputs/segment_summary.csv", index=False)

print("Saved:")
print("  ../outputs/rfm_segments.csv -", len(rfm), "customers")
print("  ../outputs/segment_summary.csv -", len(segment_summary), "segments")
print()
print(segment_summary)

Saved:
  ../outputs/rfm_segments.csv - 4338 customers
  ../outputs/segment_summary.csv - 3 segments

   Segment  customer_count  avg_recency  avg_frequency  avg_monetary  \
0  At Risk            2150       166.33           2.13        840.97   
1  Regular            1311        24.48           2.57        914.40   
2      VIP             877        13.37          12.07       6732.67   

   total_revenue  
0     1808079.53  
1     1198773.21  
2     5904555.16  
